# # manmit get data response

# # cocard manmit

## mindah ke folder berdasarkan mitra

In [ ]:
# masukin ke folder terpilih dari folder cocard di download dengan user ASUS dan
import os
import shutil
import pandas as pd

# 1. Baca data CSV kamu
df = pd.read_csv("sobatid.csv")
df['hasil']=''
df['log']=''

# 2. Ambil daftar nama file dari kolom CSV (sesuaikan nama kolomnya)
# Pastikan data berupa string dan hilangkan spasi yang tidak sengaja terikut
nama_cari = df["sobatid"].astype(str).str.strip().tolist()

# 3. Tentukan jalur (path) folder kamu
folder_sumber = r"D:\2026\SE\cocard\terpilih"
folder_tujuan = r"D:\2026\SE\cocard\terpilihv2"

# Buat folder tujuan otomatis jika belum ada
os.makedirs(folder_tujuan, exist_ok=True)

# Ambil semua daftar file asli yang ada di folder sumber sekali saja
isi_folder_sumber = os.listdir(folder_sumber)

# 2. Loop berdasarkan data di DataFrame CSV
for indeks, baris in df.iterrows():
    # Ambil kata kunci dari kolom CSV (sesuaikan nama 'nama_kolom_csv')
    kata_cari = str(baris["sobatid"]).strip()

    # Lewati jika baris kosong atau NaN
    if not kata_cari or kata_cari == "nan":
        continue

    file_ditemukan = False

    # 3. Cek ke semua file di folder sumber
    for nama_file in isi_folder_sumber:
        # Cek apakah file berupa PNG dan mengandung kata dari CSV
        if nama_file.lower().endswith(".png") and (kata_cari in nama_file):
            jalur_asal = os.path.join(folder_sumber, nama_file)
            jalur_baru = os.path.join(folder_tujuan, nama_file)

            # Pindahkan file
            shutil.move(jalur_asal, jalur_baru)
            log = f"[{kata_cari}] -> Berhasil memindahkan: {nama_file}"
            print(log)
            df.at[indeks, 'hasil'] = 'found'
            df.at[indeks, 'log'] = log


            # Perbarui daftar agar file yang sudah pindah tidak dicek lagi
            isi_folder_sumber.remove(nama_file)
            file_ditemukan = True
            break  # Keluar dari loop file jika sudah ketemu dan pindah

    # 4. Print jika tidak ditemukan
    if not file_ditemukan:
        log = f"[{kata_cari}] -> Tidak ditemukan"
        print(log)
        df.at[indeks, 'hasil'] = 'notfound'
        df.at[indeks, 'log'] = log

df.to_csv('sobatid.csv')

[510326050099] -> Tidak ditemukan
[510326050050] -> Tidak ditemukan
[510326050080] -> Tidak ditemukan
[510326050052] -> Tidak ditemukan
[510326050097] -> Tidak ditemukan
[510326050070] -> Tidak ditemukan
[510326050092] -> Tidak ditemukan
[510326050330] -> Tidak ditemukan
[510326050094] -> Tidak ditemukan
[510326050071] -> Tidak ditemukan
[510326050090] -> Tidak ditemukan
[510326050091] -> Tidak ditemukan
[510326050077] -> Tidak ditemukan
[510326050064] -> Tidak ditemukan
[510326050075] -> Tidak ditemukan
[510326050263] -> Tidak ditemukan


## generate url cek mitra

In [ ]:
id_ms = "469568"
id_mitra = "426918"
kd_survei = "SE2026"
id_kegiatan = "66"
kd_prov = "51"

In [ ]:

# def generate_url_code(t, kd_survei, id_keg, kd_prov):
def generate_url_code(instance,var=[]):
    '''var=["id_ms", "id_mitra", "kd_survei", 'id_keg', 'kd_prov']'''

    import json
    from lzstring import LZString
    allowed_keys = ["id_ms", "id_mitra", "kd_survei", 'id_keg', 'kd_prov']
    a = dict(zip(allowed_keys, var))

    # Membuat format string yang di-concat seperti di JS
    concat_str = f"{a.get('id_ms')},{a.get('id_mitra')},{a['kd_survei']},{a['id_keg']},{a['kd_prov']}"

    # Ubah string tersebut menjadi format JSON string
    # JS: JSON.stringify(...)
    json_str = json.dumps(concat_str)

    # Kompresi menggunakan LZString ke format Encoded URI Component
    lz = LZString()
    compressed = lz.compressToEncodedURIComponent(json_str)

    return f"https://mitra.bps.go.id/c/{compressed}"


In [ ]:
url_hasil = generate_url_code([id_ms, id_mitra, kd_survei, id_kegiatan, kd_prov])
url_hasil

'https://mitra.bps.go.id/c/EQFgbAnArGAcA0IBMkCMCDKBRJAGF8YY8UqwQA'

## export pdf to png and rename it

In [ ]:
import os
import csv
path_poppler_windows = r"C:\poppler-26.02.0\Library\bin" # Sesuaikan dengan versi yang Anda unduh
from pdf2image import convert_from_path

# 1. Konfigurasi File & Folder
pdf_path = "pelatihanSE/cocard-finish.pdf"
csv_path = "pelatihanSE/kepka.csv"
output_dir = "pelatihanSE/cocard-finish"

os.makedirs(output_dir, exist_ok=True)

# 2. Baca data dari CSV dan simpan ke dalam List
daftar_nama_baru = []
with open(csv_path, mode='r', encoding='utf-8') as file:
    reader = csv.DictReader(file)
    for row in reader:
        # Menggabungkan kolom id dan nama menjadi nama file baru
        # Contoh: "101_Invoice_Januari.png"
        nama_file = f"{row['nama']}_{row['sobat_id']}.png"
        daftar_nama_baru.append(nama_file)

# 3. Export PDF dan langsung namai berdasarkan CSV
print(f"Membuka PDF: {pdf_path}...")
pages = convert_from_path(pdf_path, poppler_path=path_poppler_windows)

print("Mengekstrak halaman dan mengganti nama sesuai CSV...")
for i, page in enumerate(pages):
    # Jika baris CSV lebih sedikit dari jumlah halaman PDF, gunakan nama default
    if i < len(daftar_nama_baru):
        nama_akhir = daftar_nama_baru[i]
    else:
        nama_akhir = f"halaman_{i + 1}_tanpa_nama_csv.png"
        print(f"Peringatan: Baris CSV habis. Halaman {i+1} menggunakan nama default.")

    # Simpan file langsung dengan nama baru
    path_simpan = os.path.join(output_dir, nama_akhir)
    page.save(path_simpan, 'PNG')
    print(f"Tersimpan: {nama_akhir}")

print(f"\nSelesai! {len(pages)} halaman berhasil diproses di folder '{output_dir}'.")

## or generate image data from template.png (mailmerge)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
import qrcode
import textwrap
import os

def generate_custom_document(nama, data_qr, config, output_name):
    cfg = config

    # --- 1. PROSES LOAD IMAGE & FONT ---
    try:
        # 🌟 UBAH: Gunakan mode RGBA untuk menjaga channel transparansi agar warna tidak rusak/kuning
        img = Image.open(cfg["bg_path"]).convert("RGBA")
        draw = ImageDraw.Draw(img)

        # Coba load font utama (Raleway)
        font = ImageFont.truetype(cfg['font_type'], cfg["font_size"])
        print("ℹ️ Mencoba memuat font Raleway...")
    except Exception as e:
        # 🌟 UBAH: Jika file font utama corrupt/gagal, otomatis fallback ke Arial bawaan Windows
        print(f"⚠️ Font Raleway bermasalah, otomatis dialihkan ke Arial Windows. Detail: {e}")
        try:
            font = ImageFont.truetype(r"C:\Windows\Fonts\arial.ttf", cfg["font_size"])
        except:
            # Jika Arial tidak ketemu (misal bukan di Windows), gunakan font default bawaan Pillow
            font = ImageFont.load_default()

    # --- 2. LOGIKA NAMA & GAMBAR TEKS ---
    words = nama.split()
    if len(nama) > 35:
        nama = f"{' '.join(words[:-1])} {words[-1]}."

    lines = textwrap.wrap(nama, width=15)

    x1, y1, w_box, h_box = cfg["text_box"]
    x_center = x1 + (w_box / 2)
    y_current = y1

    for line in lines:
        bbox = draw.textbbox((0, 0), line, font=font)
        line_height = bbox[3] - bbox[1]

        draw.text((x_center, y_current), line, fill="black", font=font, anchor="ma")
        y_current += line_height + 10

    # --- 3. GAMBAR QR CODE ---
    qr = qrcode.make(data_qr)

    # 🌟 UBAH: Konversi QR ke mode RGBA agar sinkron dengan mode gambar background
    qr = qr.convert("RGBA")

    qr_width = cfg["qr_box"][2]
    qr_height = cfg["qr_box"][3]
    qr = qr.resize((qr_width, qr_height))

    qx = cfg["qr_box"][0] + (qr_width - qr.size[0]) // 2
    qy = cfg["qr_box"][1] + (qr_height - qr.size[1]) // 2

    # 🌟 UBAH: Tambahkan parameter mask=qr agar warna hitam-putih QR terisolasi dan tidak berubah kuning
    img.paste(qr, (qx, qy), mask=qr)

    # Konversi balik ke RGB sebelum disimpan sebagai gambar final
    final_img = img.convert("RGB")
    final_img.save(output_name)
    print(f"✅ Sukses! Dokumen disimpan sebagai: {output_name}")

    # loerm

In [ ]:
# path
# BASE_DIR = os.path.dirname(os.path.abspath(__file__))

# Contoh config untuk template yang berbeda
templates = {
    "id_card": {
        # "bg_path": os.path.join(BASE_DIR, "pelatihanSE", "cocard-template.png"),
        "bg_path": r"D:\jim\auto-fasih-sm\pelatihanSE\cocard-template.png",
        "text_box": (138, 813, 200, 970), # start x, start y, width box, len box
        "qr_box": (460, 1148, 330, 330),
        "font_size": 27,
        # "font_type": os.path.join(BASE_DIR, "pelatihanSE", "raleway-medium.ttf")
        "font_type": r"D:\jim\auto-fasih-sm\pelatihanSE\raleway-medium.ttf"
        # "font_type": "arial.ttf"
    }
}

# Cara Pakai:
generate_custom_document("Jey Neutron", "link_data", templates["id_card"], "hasil_cocard.png")


ℹ️ Mencoba memuat font Raleway...
✅ Sukses! Dokumen disimpan sebagai: hasil_cocard.png


In [ ]:
from PIL import Image, ImageDraw, ImageFont
import qrcode
import textwrap

def gen_mergemail(nama, data_qr, config, output_name):
    cfg = config

    try:
        # Load background RGBA agar warna QR terkunci hitam-putih
        img = Image.open(cfg["bg_path"]).convert("RGBA")
        draw = ImageDraw.Draw(img)
        font = ImageFont.truetype(cfg['font_type'], cfg["font_size"])
    except Exception as e:
        print(f"❌ GAGAL LOAD FILE! Detail: {e}")
        return

    # 1. Bersihkan String Nama
    nama_clean = str(nama).strip().upper()

    x1, y1, w_box, h_box = cfg["text_box"]
    x_center = img.width / 2
    y_floor = y1

    # Gunakan textwrap dengan width lebih besar agar kata tidak terpotong per huruf
    if len(nama_clean) > 35:
        nama_potong = nama_clean[:35]
        words = nama_potong.split()
        if len(words) > 1:
            all_but_last = " ".join(words[:-1])
            nama_final = f"{all_but_last} {words[-1][0]}."
        else:
            nama_final = nama_clean[:20] # Fallback jika hanya 1 kata panjang
        # Jika lebih dari 35 karakter, paksa potong lebih pendek agar otomatis jadi 2 baris yang seimbang
        lines = textwrap.wrap(nama_final, width=20)
    else:
        # Jika di bawah 35 karakter, berikan ruang lebar agar tetap aman dalam 1 baris
        lines = textwrap.wrap(nama_clean, width=30)
    text_to_draw = "\n".join(lines)

    y_start = y1 - (cfg["font_size"] * (len(lines) - 1) + 20)

    # UBAH: Menggunakan anchor="ma" (Middle-Top) dikombinasikan dengan hitungan y_start dinamis di atas
    draw.text((x_center, y_start), text_to_draw, fill="black", font=font, anchor="ma", align="center")

    # 3. Gambar QR Code (Hitam-Putih Bersih)
    qr = qrcode.make(data_qr).convert("RGBA")
    qr_width = cfg["qr_box"][2]
    qr_height = cfg["qr_box"][3]
    qr = qr.resize((qr_width, qr_height))

    qx = cfg["qr_box"][0] + (qr_width - qr.size[0]) #// 2
    qy = cfg["qr_box"][1] + (qr_height - qr.size[1]) #// 2
    img.paste(qr, (qx, qy), mask=qr)

    # Simpan instan dalam hitungan milidetik
    final_img = img.convert("RGB")
    final_img.save(output_name)
    print(f"⚡ Sukses Generate: {output_name}")

In [ ]:
# --- KONFIGURASI BARU ---
templates = {
    "id_card": {
        # "bg_path": r"D:\jim\auto-fasih-sm\pelatihanSE\cocard-template.png",
        "bg_path": r"pelatihanSE\cocard-template.png",
        "text_box": (200, 955, 600, 970), # start x, start y, width box, height box; sementara yg kepake baru y aja
        "qr_box": (450, 1148, 330, 330),
        "font_size": 80,
        # "font_type": r"C:\Windows\Fonts\arial.ttf"
        # "font_type": r"D:\jim\auto-fasih-sm\raleway-medium.ttf"
        "font_type": r"pelatihanSE\Raleway-SemiBold.ttf"
    }
}

# try Jalankan fungsi super cepat
gen_mergemail("Lorem ipsum dolor", "link_data", templates["id_card"], "temphasil.png")
gen_mergemail("Lorem ipsum dolor sit amet lorem loreman", "https://mitra.bps.go.id/c/EQdhBYCZwNgGgMwE4AcyUjgZQKKQAyTwzwCsAjMEA", templates["id_card"], "temphasil2.png")



⚡ Sukses Generate: temphasil.png
⚡ Sukses Generate: temphasil2.png


# # fasih assign


## get error json as csv

In [ ]:
import pandas as pd
import json

# 1. Struktur data JSON Anda (sudah dilengkapi penutupnya)
with open("errors.json", "r", encoding="utf-8") as file:
    raw_json = json.load(file)

# 2. Ambil data langsung dari dalam key 'content'
target_data = raw_json["data"]["content"]

# 3. Ubah menjadi DataFrame
df = pd.DataFrame(target_data)

# 4. Opsional: Bersihkan kolom 'users' agar tidak membawa tanda kurung siku [] di CSV
df["users"] = df["users"].apply(lambda x: ", ".join(x) if isinstance(x, list) else x)

# 5. Simpan ke file CSV
df.to_csv("error.json.csv", index=False)
print("Data JSON berhasil dikonversi ke error.json.csv!")

Data JSON berhasil dikonversi ke error.json.csv!


## get mitra assigned

In [ ]:
import json
import pandas as pd

# 1. Membaca file JSON
# (Asumsi teks JSON Anda dibungkus tanda kurung kurawal agar valid)
with open("by-user.json", "r") as file:
    payload = json.load(file)

# 2. Mengambil data utama dari dalam key 'data' -> 'content'
content_data = payload["data"]["content"]

# 3. Mengubah ke DataFrame dan membongkar list 'regions' di dalamnya
df = pd.json_normalize(
    content_data,
    record_path=["regions"],
    meta=["userId", "roleId", "totalRegions", "username"],
)

# 4. Memilih dan mengurutkan kolom sesuai kebutuhan Anda
df_hasil = df[["userId", "roleId", "totalRegions", "regionCode", "username"]]

# Tampilkan hasil
df_hasil


# metabase scrap

## sebali

In [79]:
import gspread
import requests
import pandas as pd
from google.colab import auth
from google.auth import default

# 1. Log in ke Akun Google Anda
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. Ambil data dari URL Anda
# Silakan ganti dengan URL asli yang menghasilkan data JSON tersebut
url = "https://metabase.statsbali.id/api/public/dashboard/e66b6814-a55a-4f92-b50e-5d459876e880/dashcard/147/card/210?parameters=%5B%7B%22type%22%3A%22string%2F%3D%22%2C%22value%22%3Anull%2C%22id%22%3A%22800e7598%22%2C%22target%22%3A%5B%22dimension%22%2C%5B%22field%22%2C%22kab_filter%22%2C%7B%22base-type%22%3A%22type%2FText%22%7D%5D%2C%7B%22stage-number%22%3A0%7D%5D%7D%5D"
response = requests.get(url)
json_data1 = response.json()  # Mengambil respons dalam bentuk JSON

# Mengambil list data dari dalam key 'data' -> 'rows'
rows = json_data1["data"]["rows"]

# 3. Ubah data menjadi Tabel Pivot menggunakan Pandas
df_raw = pd.DataFrame(rows, columns=["Kode", "Daerah", "Status", "Jumlah"])
df_raw['Daerah'] = df_raw['Kode'].str[2:]+' - '+df_raw['Daerah'] # rapiin index
# Pastikan data Jumlah adalah angka, jika null/kosong otomatis jadi 0
df_raw["Jumlah"] = pd.to_numeric(df_raw["Jumlah"], errors='coerce').fillna(0).astype(int)
# Jika isi kolom Status ternyata kosong (None / NaN) atau berupa teks kosong, ubah namanya jadi "NULL"
df_raw["Status"] = df_raw["Status"].fillna("NULL").replace("", "NULL")

# Bikin tabel pivot awal berdasarkan semua status yang masuk dari API
df_pivot = df_raw.pivot_table(index="Daerah", columns="Status", values="Jumlah", aggfunc="sum")
df_pivot = df_pivot.fillna(0).astype(int)

# --- PROSES PEMISAHAN STATUS UTAMA VS STATUS LAINNYA ---
status_utama = ["APPROVED", "DRAFT", "OPEN", "REJECTED", "REVOKED", "SUBMITTED"]
# Cari tahu status apa saja yang muncul di data tapi tidak ada di daftar status utama (termasuk "NULL")
status_lainnya = [col for col in df_pivot.columns if col not in status_utama]
# Buat kolom penampung dinamis untuk menampung total dari semua status lainnya / null tersebut
if status_lainnya:
    df_pivot["kolomkosong"] = df_pivot[status_lainnya].sum(axis=1)
else:
    df_pivot["kolomkosong"] = 0  # Isinya 0 jika semua data berstatus normal

# Pastikan semua 6 status utama tetap ada di tabel (jika di data aslinya absen, otomatis diisi 0)
for status in status_utama:
    if status not in df_pivot.columns:
        df_pivot[status] = 0

# Susun urutan kolomnya sesuai permintaan Anda
urutan_kolom_akhir = status_utama + ["kolomkosong"]
df_pivot = df_pivot[urutan_kolom_akhir]
# Tambahkan kolom Row totals hasil penjumlahan dari semua status utama + kolom penampung
df_pivot["Row totals"] = df_pivot.sum(axis=1)
# Kembalikan kolom "Daerah" menjadi kolom biasa
df_pivot = df_pivot.reset_index()

# 4. Siapkan data untuk dikirim ke Google Sheets
header = [df_pivot.columns.values.tolist()]
data_rows = df_pivot.values.tolist()
final_data = header + data_rows

# 5. Buka Google Sheet dan Tulis Datanya
# Ganti dengan nama file Google Sheet Anda
spreadsheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/1EPdxbPZipQRTH0ZQDHE-hFi_oU2Xn9UfGyblzYwmZ6U/edit?gid=982428334#gid=982428334")
worksheet = spreadsheet.worksheet("Rekap_region")

# Tulis tabel baru mulai dari sel kiri atas
worksheet.update(range_name="c6", values=final_data, raw=False)

print("Sukses! Tabel pivot dan totalnya sudah rapi di Google Sheets.")


Sukses! Tabel pivot dan totalnya sudah rapi di Google Sheets.


In [74]:
df_pivot

Status,Daerah,APPROVED,DRAFT,OPEN,REJECTED,REVOKED,SUBMITTED,kolomkosong,Row totals
0,01 - JEMBRANA,1938,2913,128169,1132,10,8519,0,142681
1,02 - TABANAN,2938,3151,179777,1117,14,14542,1,201540
2,03 - BADUNG,1927,5879,185024,603,38,10909,1,204381
3,04 - GIANYAR,1783,5917,193311,514,14,7617,0,209156
4,05 - KLUNGKUNG,1562,2875,70534,430,9,4312,1,79723
5,06 - BANGLI,547,2834,103494,204,2,7022,2,114105
6,07 - KARANGASEM,2663,3952,182915,932,18,13089,1,203570
7,08 - BULELENG,7374,6200,278372,1577,52,21844,2,315421
8,71 - DENPASAR,1867,5072,276397,364,10,10710,1,294421


## date latest update

In [83]:
import gspread
import requests
from google.colab import auth
from google.auth import default

# 1. Log in ke Akun Google Anda
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. Ambil data dari URL Anda
# Ganti dengan URL asli yang menghasilkan data tanggal tersebut
url = "https://metabase.statsbali.id/api/public/dashboard/e66b6814-a55a-4f92-b50e-5d459876e880/dashcard/142/card/201?parameters=%5B%5D"
response = requests.get(url)
json_data = response.json()

# Mengambil isi dari key 'data' -> 'rows'
rows = json_data["data"]["rows"]

# Mengambil data baris pertama saja (index ke-0)
# Hasilnya berupa teks tanggal: "2026-06-17T19:32:57+08:00"
data_tanggal = rows[0][0]
data_tanggal = data_tanggal.replace("T", " ")[:19]

# 3. Buka Google Sheet Anda
# Ganti dengan nama file Google Sheet Anda
# spreadsheet = gc.open("YOUR_SPREADSHEET_NAME")
worksheet = spreadsheet.worksheet("raw_metabase")

# 4. Update data ke cell tertentu tanpa merusak format sheet
target_cell = "e1"  # Ganti dengan posisi cell tempat menaruh tanggalnya

# Menggunakan raw=False agar Google Sheet mempertahankan format asli cell tersebut
worksheet.update(range_name=target_cell, values=[[data_tanggal]], raw=False)

print(f"Sukses! Nilai tanggal '{data_tanggal}' sudah di-update di cell {target_cell} tanpa mengubah format asli sheet.")


Sukses! Nilai tanggal '2026-06-17T19:32:57+08:00' sudah di-update di cell e1 tanpa mengubah format asli sheet.


In [82]:
data_tanggal

'2026-06-17T19:32:57+08:00'

## sebadung

In [84]:
url = "https://metabase.statsbali.id/api/public/pivot/dashboard/e66b6814-a55a-4f92-b50e-5d459876e880/dashcard/143/card/203?parameters=%5B%7B%22type%22%3A%22string%2F%3D%22%2C%22value%22%3A%5B%2203%20-%20BADUNG%22%5D%2C%22id%22%3A%22800e7598%22%2C%22target%22%3A%5B%22dimension%22%2C%5B%22field%22%2C%22kab_filter%22%2C%7B%22base-type%22%3A%22type%2FText%22%7D%5D%2C%7B%22stage-number%22%3A0%7D%5D%7D%5D"
response = requests.get(url)
json_data = response.json()  # Mengambil respons dalam bentuk JSON

In [85]:
import gspread
import requests
import pandas as pd
from google.colab import auth
from google.auth import default

# 1. Log in ke Akun Google Anda
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# 2. Ambil data dari URL Anda
# Silakan ganti dengan URL asli yang menghasilkan data JSON tersebut
# url = "https://metabase.statsbali.id/api/public/pivot/dashboard/e66b6814-a55a-4f92-b50e-5d459876e880/dashcard/143/card/203?parameters=%5B%7B%22type%22%3A%22string%2F%3D%22%2C%22value%22%3A%5B%2203%20-%20BADUNG%22%5D%2C%22id%22%3A%22800e7598%22%2C%22target%22%3A%5B%22dimension%22%2C%5B%22field%22%2C%22kab_filter%22%2C%7B%22base-type%22%3A%22type%2FText%22%7D%5D%2C%7B%22stage-number%22%3A0%7D%5D%7D%5D"
# response = requests.get(url)
# json_data = response.json()  # Mengambil respons dalam bentuk JSON

# Mengambil list data dari dalam key 'data' -> 'rows'
rows = json_data["data"]["rows"]

# 1. Buat DataFrame awal dari data rows API Anda
df_raw = pd.DataFrame(rows, columns=["kab", "kec", "desa", "sls", "subsls", "Status", "val", "Jumlah"])

# --- PROSES PEMBERSIHAN DATA REKAP (TIDAK MENGHAPUS STATUS NULL DI SINI) ---
# A. Buang data rekap (Baris yang subsls-nya tidak ada)
df_raw["subsls"] = df_raw["subsls"].replace(["None", "NaN", ""], pd.NA)
df_raw = df_raw.dropna(subset=["subsls"])

# B. Isi Status yang kosong dengan teks "NULL" agar seragam saat dihitung
df_raw["Status"] = df_raw["Status"].fillna("NULL").replace(["", "None"], "NULL")
# ---------------------------------------------------------------------------

# 2. Pastikan data Jumlah adalah angka bulat
df_raw["Jumlah"] = pd.to_numeric(df_raw["Jumlah"], errors='coerce').fillna(0).astype(int)

# 3. Definisikan 6 status utama Anda yang murni
status_utama = ["APPROVED", "DRAFT", "OPEN", "REJECTED", "REVOKED", "SUBMITTED"]

# 4. Ambil data yang murni termasuk dalam status utama saja
df_status_utama = df_raw[df_raw["Status"].isin(status_utama)].copy()

# --- PERBAIKAN KUNCI DI SINI: MENGURUNG DATA LUAR / STATUS NULL ASLI ---
# Kita ambil baris yang Statusnya BUKAN status utama DAN kolom val-nya BUKAN 32
# Ini trik rahasia untuk membuang duplikat angka kembar (val=32) tapi TETAP menjaga angka 1 (val=0 atau lainnya)
df_status_lainnya = df_raw[(~df_raw["Status"].isin(status_utama)) & (df_raw["val"] != 32)].copy()

# 5. BIKIN TABEL PIVOT DARI DATA STATUS UTAMA
df_pivot = df_status_utama.pivot_table(
    index=["kec", "desa", "sls", "subsls"],
    columns="Status",
    values="Jumlah",
    aggfunc="sum"
)
df_pivot = df_pivot.fillna(0).astype(int)

# 6. HITUNG KOLOM KOSONG (Sekarang nilai 1 pasti ikut terhitung masuk)
if not df_status_lainnya.empty:
    df_lain_pivot = df_status_lainnya.pivot_table(
        index=["kec", "desa", "sls", "subsls"],
        values="Jumlah",
        aggfunc="sum"
    )
    df_pivot["kolomkosong"] = df_pivot.index.map(df_lain_pivot["Jumlah"]).fillna(0).astype(int)
else:
    df_pivot["kolomkosong"] = 0

# 7. PASTIKAN SEMUA 6 STATUS UTAMA TETAP TAMPIL DI KOLOM SHEET
for status in status_utama:
    if status not in df_pivot.columns:
        df_pivot[status] = 0

# 8. HITUNG ROW TOTALS (6 status utama + kolomkosong)
df_pivot["Row totals"] = df_pivot[status_utama].sum(axis=1) + df_pivot["kolomkosong"]

# 9. SUSUN URUTAN KOLOM AKHIR SEBELUM DIKIRIM
urutan_kolom_akhir = status_utama + ["kolomkosong", "Row totals"]
df_pivot = df_pivot[urutan_kolom_akhir]

# Langkah A: Reset index LEBIH DULU agar kolom wilayah pecah terpisah secara normal
df_pivot = df_pivot.reset_index()

# Langkah B: Hitung jumlah total ke bawah (hanya untuk kolom angka)
total_bawah = df_pivot.sum(numeric_only=True)

# Langkah C: Tambahkan baris total tersebut ke posisi paling bawah tabel
df_pivot.loc[len(df_pivot)] = total_bawah

# Langkah D: Isi nama kolom wilayah pada baris terakhir agar rapi
df_pivot.iloc[-1, df_pivot.columns.get_loc("kec")] = "TOTAL KESELURUHAN"
df_pivot.iloc[-1, df_pivot.columns.get_loc("desa")] = ""
df_pivot.iloc[-1, df_pivot.columns.get_loc("sls")] = ""
df_pivot.iloc[-1, df_pivot.columns.get_loc("subsls")] = ""

# 4. Siapkan data untuk dikirim ke Google Sheets
header = [df_pivot.columns.values.tolist()]
data_rows = df_pivot.values.tolist()
final_data = header + data_rows

# 5. Buka Google Sheet dan Tulis Datanya
# Ganti dengan nama file Google Sheet Anda
spreadsheet = gc.open_by_url("https://docs.google.com/spreadsheets/d/1EPdxbPZipQRTH0ZQDHE-hFi_oU2Xn9UfGyblzYwmZ6U/edit?gid=982428334#gid=982428334")
worksheet = spreadsheet.worksheet("raw_metabase")

# # Tulis tabel baru mulai dari sel kiri atas
worksheet.update(range_name="c2", values=final_data, raw=False)

print("Sukses! Tabel pivot dan totalnya sudah rapi di Google Sheets.")


Sukses! Tabel pivot dan totalnya sudah rapi di Google Sheets.


In [78]:
df_pivot

Status,kec,desa,sls,subsls,APPROVED,DRAFT,OPEN,REJECTED,REVOKED,SUBMITTED,kolomkosong,Row totals
0,000 - -,000 - -,0000 - -,00,0.0,1.0,114.0,0.0,0.0,121.0,0.0,236.0
1,010 - KUTA SELATAN,001 - PECATU,0001 - BANJAR KANGIN,01,0.0,0.0,133.0,0.0,0.0,0.0,0.0,133.0
2,010 - KUTA SELATAN,001 - PECATU,0001 - BANJAR KANGIN,02,0.0,0.0,57.0,0.0,0.0,0.0,0.0,57.0
3,010 - KUTA SELATAN,001 - PECATU,0001 - BANJAR KANGIN,03,0.0,2.0,51.0,0.0,0.0,7.0,0.0,60.0
4,010 - KUTA SELATAN,001 - PECATU,0001 - BANJAR KANGIN,04,4.0,1.0,29.0,1.0,0.0,13.0,0.0,48.0
...,...,...,...,...,...,...,...,...,...,...,...,...
2270,060 - PETANG,007 - BELOK/SIDAN,0007 - BANJAR BON,02,0.0,0.0,85.0,0.0,0.0,0.0,0.0,85.0
2271,060 - PETANG,007 - BELOK/SIDAN,0008 - BANJAR JEMPANANG,00,0.0,14.0,202.0,0.0,0.0,16.0,0.0,232.0
2272,060 - PETANG,007 - BELOK/SIDAN,0009 - BANJAR SEKARMUKTI,01,0.0,0.0,140.0,0.0,0.0,0.0,0.0,140.0
2273,060 - PETANG,007 - BELOK/SIDAN,0009 - BANJAR SEKARMUKTI,02,0.0,0.0,115.0,0.0,0.0,0.0,0.0,115.0


## all join

In [ ]:
import gspread
import requests
import pandas as pd
from google.colab import auth
from google.auth import default

# ==========================================
# 1. AUTENTIKASI GOOGLE (Cukup Sekali)
# ==========================================
auth.authenticate_user()
creds, _ = default()
gc = gspread.authorize(creds)

# Buka Spreadsheet utama
SPREADSHEET_URL = "https://docs.google.com/spreadsheets/d/1EPdxbPZipQRTH0ZQDHE-hFi_oU2Xn9UfGyblzYwmZ6U/edit?gid=982428334#gid=982428334"
spreadsheet = gc.open_by_url(SPREADSHEET_URL)

# ==========================================
# 2. FUNGSI UTAMA (GENERAL FUNCTION)
# ==========================================
def process_metabase_to_sheets(url, sheet_name, cell_target, mode="pivot_wilayah", columns_def=None):
    """
    Fungsi general untuk mengambil data dari API Metabase, memprosesnya dengan Pandas,
    dan memperbarui data ke Google Sheets berdasarkan mode tertentu.
    
    Mode yang tersedia:
    - 'pivot_detail' : Untuk Case 1 (Hierarki wilayah Kec, Desa, SLS, SubSLS + Total Bawah)
    - 'pivot_region' : Untuk Case 2 (Hierarki Kode/Daerah Gabungan)
    - 'single_value' : Untuk Case 3 (Mengambil nilai tunggal seperti Tanggal)
    """
    print(f"Memproses sheet '{sheet_name}' pada cell '{cell_target}' dengan mode '{mode}'...")
    
    # A. Fetch Data dari API
    response = requests.get(url)
    json_data = response.json()
    rows = json_data["data"]["rows"]
    
    worksheet = spreadsheet.worksheet(sheet_name)
    status_utama = ["APPROVED", "DRAFT", "OPEN", "REJECTED", "REVOKED", "SUBMITTED"]
    
    # B. Pemrosesan Data Berdasarkan Mode
    if mode == "single_value":
        # Case 3: Ambil tanggal, bersihkan format teks
        data_tanggal = rows[0][0].replace("T", " ")[:19]
        worksheet.update(range_name=cell_target, values=[[data_tanggal]], raw=False)
        print(f"Sukses update tanggal: {data_tanggal}")
        return

    # Jika bukan single_value, ubah ke DataFrame awal
    df_raw = pd.DataFrame(rows, columns=columns_def)
    df_raw["Jumlah"] = pd.to_numeric(df_raw["Jumlah"], errors='coerce').fillna(0).astype(int)
    df_raw["Status"] = df_raw["Status"].fillna("NULL").replace(["", "None"], "NULL")

    if mode == "pivot_detail":
        # Case 1: Filter rekap & penanganan khusus 'val'
        df_raw["subsls"] = df_raw["subsls"].replace(["None", "NaN", ""], pd.NA)
        df_raw = df_raw.dropna(subset=["subsls"])
        
        df_status_utama = df_raw[df_raw["Status"].isin(status_utama)].copy()
        df_status_lainnya = df_raw[(~df_raw["Status"].isin(status_utama)) & (df_raw["val"] != 32)].copy()
        
        # Buat pivot utama
        df_pivot = df_status_utama.pivot_table(
            index=["kec", "desa", "sls", "subsls"], columns="Status", values="Jumlah", aggfunc="sum"
        ).fillna(0).astype(int)
        
        # Hitung kolom kosong
        if not df_status_lainnya.empty:
            df_lain_pivot = df_status_lainnya.pivot_table(
                index=["kec", "desa", "sls", "subsls"], values="Jumlah", aggfunc="sum"
            )
            df_pivot["kolomkosong"] = df_pivot.index.map(df_lain_pivot["Jumlah"]).fillna(0).astype(int)
        else:
            df_pivot["kolomkosong"] = 0
            
        # Pastikan kolom status lengkap
        for status in status_utama:
            if status not in df_pivot.columns:
                df_pivot[status] = 0
                
        df_pivot["Row totals"] = df_pivot[status_utama].sum(axis=1) + df_pivot["kolomkosong"]
        df_pivot = df_pivot[status_utama + ["kolomkosong", "Row totals"]].reset_index()
        
        # Tambahkan Baris TOTAL KESELURUHAN di paling bawah
        total_bawah = df_pivot.sum(numeric_only=True)
        df_pivot.loc[len(df_pivot)] = total_bawah
        df_pivot.iloc[-1, df_pivot.columns.get_loc("kec")] = "TOTAL KESELURUHAN"
        for col in ["desa", "sls", "subsls"]:
            df_pivot.iloc[-1, df_pivot.columns.get_loc(col)] = ""

    elif mode == "pivot_region":
        # Case 2: Penggabungan kode dan nama daerah
        df_raw['Daerah'] = df_raw['Kode'].str[2:] + ' - ' + df_raw['Daerah']
        
        df_pivot = df_raw.pivot_table(index="Daerah", columns="Status", values="Jumlah", aggfunc="sum").fillna(0).astype(int)
        
        status_lainnya = [col for col in df_pivot.columns if col not in status_utama]
        df_pivot["kolomkosong"] = df_pivot[status_lainnya].sum(axis=1) if status_lainnya else 0
        
        for status in status_utama:
            if status not in df_pivot.columns:
                df_pivot[status] = 0
                
        df_pivot = df_pivot[status_utama + ["kolomkosong"]]
        df_pivot["Row totals"] = df_pivot.sum(axis=1)
        df_pivot = df_pivot.reset_index()

    # C. Kirim Hasil Akhir ke Google Sheets
    final_data = [df_pivot.columns.values.tolist()] + df_pivot.values.tolist()
    worksheet.update(range_name=cell_target, values=final_data, raw=False)
    print(f"Sukses memperbarui tabel di {sheet_name} ({cell_target}).\n")



In [ ]:
# ==========================================
# 3. EKSEKUSI KETIGA CASE
# ==========================================
import time
# --- CASE 1: Detail SLS (raw_metabase) ---
url_case1 = "https://metabase.statsbali.id/api/public/pivot/dashboard/e66b6814-a55a-4f92-b50e-5d459876e880/dashcard/143/card/203?parameters=%5B%7B%22type%22%3A%22string%2F%3D%22%2C%22value%22%3A%5B%2203%20-%20BADUNG%22%5D%2C%22id%22%3A%22800e7598%22%2C%22target%22%3A%5B%22dimension%22%2C%5B%22field%22%2C%22kab_filter%22%2C%7B%22base-type%22%3A%22type%2FText%22%7D%5D%2C%7B%22stage-number%22%3A0%7D%5D%7D%5D"
columns_case1 = ["kab", "kec", "desa", "sls", "subsls", "Status", "val", "Jumlah"]

process_metabase_to_sheets(
    url=url_case1,
    sheet_name="raw_metabase",
    cell_target="c2",
    mode="pivot_detail",
    columns_def=columns_case1
)
time.sleep(1)
# --- CASE 2: Rekap Region (Rekap_region) ---
url_case2 = "https://metabase.statsbali.id/api/public/dashboard/e66b6814-a55a-4f92-b50e-5d459876e880/dashcard/147/card/210?parameters=%5B%7B%22type%22%3A%22string%2F%3D%22%2C%22value%22%3Anull%2C%22id%22%3A%22800e7598%22%2C%22target%22%3A%5B%22dimension%22%2C%5B%22field%22%2C%22kab_filter%22%2C%7B%22base-type%22%3A%22type%2FText%22%7D%5D%2C%7B%22stage-number%22%3A0%7D%5D%7D%5D"
columns_case2 = ["Kode", "Daerah", "Status", "Jumlah"]

process_metabase_to_sheets(
    url=url_case2,
    sheet_name="Rekap_region",
    cell_target="c6",
    mode="pivot_region",
    columns_def=columns_case2
)
time.sleep(1)

# --- CASE 3: Ambil Tanggal Update (raw_metabase cell E1) ---
url_case3 = "https://metabase.statsbali.id/api/public/dashboard/e66b6814-a55a-4f92-b50e-5d459876e880/dashcard/142/card/201?parameters=%5B%5D"

process_metabase_to_sheets(
    url=url_case3,
    sheet_name="raw_metabase",
    cell_target="e1",
    mode="single_value"
)
time.sleep(1)